# Handling Missing Data

---

## What are we learning?

Real-world datasets often arrive with gaps—NaNs, blanks, or question marks. In this notebook you’ll learn how to detect, quantify, and fix missing values with pandas and scikit-learn so your models train on clean, complete data.

## The idea in plain English

Think of a spreadsheet where some cells are empty. Instead of throwing away entire rows or columns, we can “fill the potholes” by inserting sensible values (mean, median, or a learned guess) so the road (dataset) is smooth for algorithms to drive on.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

print('Setup done!')

## Step 1 — Load data

In [ ]:
# Toy dataset: 200 passengers with missing age, cabin, and embarked info
titanic = sns.load_dataset('titanic').loc[:, ['survived','pclass','sex','age','sibsp','parch','fare','embarked']].copy()
titanic.loc[::7, 'age'] = np.nan      # every 7th passenger
titanic.loc[::11, 'embarked'] = np.nan # every 11th passenger
print(titanic.head())
titanic.isna().sum()

## Step 2 — Apply Handling Missing Data

In [ ]:
# Strategy 1: Drop rows with any missing values
dropped = titanic.dropna()
print('Rows after dropna:', dropped.shape[0])

# Strategy 2: Mean imputation for numeric, most-frequent for categorical
num_cols = ['age','fare']
cat_cols = ['embarked']

imputer_num = SimpleImputer(strategy='mean')
imputer_cat = SimpleImputer(strategy='most_frequent')

titanic_imputed = titanic.copy()
titanic_imputed[num_cols] = imputer_num.fit_transform(titanic[num_cols])
titanic_imputed[cat_cols] = imputer_cat.fit_transform(titanic[cat_cols])

print('Missing values after imputation:')
print(titanic_imputed.isna().sum())

## Step 3 — Visualise

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4))

# Before
titanic.isna().sum().plot(kind='bar', ax=ax[0], color='coral')
ax[0].set_title('Missing counts BEFORE')
ax[0].set_ylabel('# missing')

# After
titanic_imputed.isna().sum().plot(kind='bar', ax=ax[1], color='seagreen')
ax[1].set_title('Missing counts AFTER')
ax[1].set_ylabel('# missing')

plt.tight_layout()
plt.show()

## Results & interpretation

In [ ]:
print('Original dataset shape:', titanic.shape)
print('Rows lost by dropping:', titanic.shape[0] - dropped.shape[0])
print('Mean age used for imputation:', imputer_num.statistics_[0])
print('Most common embark port used:', imputer_cat.statistics_[0])

## Summary

- `isna()` and `sum()` quickly reveal how many values are missing per column.
- `dropna()` is easy but can discard too much data.
- `SimpleImputer` fills gaps with mean/median/most-frequent without deleting rows.
- Always inspect the chosen imputation value to ensure it makes business sense.

## Exercises

1. Replace missing ages with the median instead of the mean and compare the two distributions with histograms.
2. Use `IterativeImputer` (experimental) to predict missing ages from pclass, fare, and sibsp; print the new mean age.
3. Create a binary column `age_was_missing` that flags rows whose age was originally NaN, then train a logistic regression to see if missingness predicts survival.

In [ ]:
# Your code here